# 重新归一化散射参数

本示例演示如何使用 skrf 重新归一化网络对象的散射参数，使其适应新的端口阻抗。虽然这个例子很简单，但它首先创建一个 50 欧姆的匹配负载，然后将其重新归一化到 25 欧姆的环境中，从而产生 1/3 的反射系数。

好的，我们开始吧。

In [ ]:
import skrf as rf
from skrf.instances import wr10

%matplotlib inline

rf.stylely()

# this is just for plotting junk
kw = dict(draw_labels=True, marker = 'o', markersize = 10)

创建一个单端口理想匹配网络（使用预定义的 wr10 介质类作为示例）。

In [ ]:
match_at_50 = wr10.match()

请注意，对于此 `skrf.Network`，其默认的 `z0`（特性阻抗）值为一个常数 50 欧姆。

In [ ]:
match_at_50

将其反射系数绘制在史密斯圆图上，显示其阻抗匹配良好。

In [ ]:
match_at_50.plot_s_smith(**kw)

现在，将端口阻抗从 50 重新归一化为 25，因此之前 50 欧姆的负载现在产生一个反射系数，其值为：$$ \Gamma^{'} = \frac{50-25}{50+25} = \frac{25}{75} = .333 $$在史密斯图上绘制归一化后的响应。

In [ ]:
match_at_50.renormalize(25)
match_at_50.plot_s_smith(**kw)

## 複阻抗

如果您想尝试，也可以重新归一化到复数端口阻抗。例如，如果将阻抗归一化为 50j，预期结果如下：$$\Gamma^{'} = \frac{50-50j}{50+50j} = 50\frac{1-j}{1+j} = -50j$$但是，在绘制史密斯圆图时，会发现结果与预期不符：

In [ ]:
match_at_50 = wr10.match()
match_at_50.renormalize(50j)  # same as renormalize(50j, s_def='power')
match_at_50.plot_s_smith(**kw)  # expect -1j

这是因为 scikit-rf 的默认行为是使用“功率波”散射参数定义（因为它是在 CAD 软件中最常用的定义）。但是，“功率波”定义在某些情况下[已知会失效](https://www.nist.gov/system/files/documents/2017/05/09/MicrowaveCircuitTheory-proof.pdf)。因此，scikit-rf 也实现了“伪波”散射参数定义，但您需要使用 `s_def` 参数来指定它：

In [ ]:
match_at_50 = wr10.match()
match_at_50.renormalize(50j, s_def='pseudo')
match_at_50.plot_s_smith(**kw)  # expect -1j

这会得到预期的结果。